# Inspect one model

**Question: where does this model make mistakes?**

1. Export a report with `python3 scripts/evaluate.py MODEL DATASET --output results/RUN`.
2. Set `RUN` below and run all cells.
3. Read the class scores and confusion matrix; record the main errors at the end.

This notebook reads one saved report. It does not train a model or run predictions.
Use validation during development and the reserved test set after model selection.
To compare several models on validation, open [comparison.ipynb](comparison.ipynb).


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RUN = ROOT / "results" / "RUN1"  # --- Select here an evaluation run ---

path = RUN / "metrics.json"
if not path.is_file():
    raise FileNotFoundError(f"No metrics at {path}. Run scripts/evaluate.py first.")
metrics = json.loads(path.read_text())
labels = metrics["confusion_matrix"]["labels"]
print(f"Run: {RUN.name} | Positions: {metrics['samples']:,}")
print(f"Accuracy: {metrics['accuracy']:.2%} | Macro F1: {metrics['macro_f1']:.3f}")
print(f"Analyzer elapsed time: {metrics['prediction_seconds']:.2f} s (includes loading)")
for name in ("model", "dataset"):
    print(f"{name.capitalize()}: {metrics['artifacts'][name]['path']}")


## Scores by class

Precision: how often predictions of a class are correct. Recall: how many actual
positions of that class were found. F1 combines both. Undefined scores are zero;
macro F1 includes all three classes, even those with no examples.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
width = 0.24
for offset, key in enumerate(("precision", "recall", "f1")):
    positions = [i + (offset - 1) * width for i in range(len(labels))]
    bars = ax.bar(positions, [metrics["per_class"][label][key] for label in labels],
                  width, label=key.capitalize())
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
ax.set_xticks(range(len(labels)),
              [f"{label}\nn={metrics['per_class'][label]['support']:,}" for label in labels])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title(f"Per-class performance — {RUN.name}")
ax.legend(loc="upper right", ncols=3)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## Confusion matrix

Rows are actual classes; columns are predictions. The diagonal shows correct
predictions. Row percentages make errors comparable across class sizes.
A row with no examples is shown as zeros.


In [ ]:
counts = metrics["confusion_matrix"]["counts"]
percentages = [[100 * value / sum(row) if sum(row) else 0 for value in row]
               for row in counts]
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
heatmap = ax.imshow(percentages, cmap="Blues", vmin=0, vmax=100)
for row in range(len(labels)):
    for column in range(len(labels)):
        percent = percentages[row][column]
        ax.text(column, row, f"{counts[row][column]:,}\n{percent:.1f}%",
                ha="center", va="center", color="white" if percent > 50 else "black")
ax.set_xticks(range(len(labels)), labels)
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title(f"Confusion matrix — {RUN.name}")
fig.colorbar(heatmap, ax=ax, label="% of actual class")
plt.show()


## Interpretation

Record the weakest class and the main confusion here. For an untrained model,
these results only check the evaluation pipeline.
